# Knee MRI: EDA to baseline, decision by decision

RSNA Knee Abnormality Detection asks, for one knee MRI exam, for the probability
that each of **12 findings** is present. The score is the mean AUC over the 12
findings (macro AUC).

This notebook walks through what the data actually looks like, and how each
observation pins down a piece of the baseline. In this competition a surprising
number of things get decided before you touch a model.

What you'll get out of it:

- where the real difficulty is (it isn't the image processing)
- which provided columns you cannot use as-is
- how to split validation so it means something
- how all of the above fixes the baseline configuration

At the end I walk through the path to **public LB 0.898**, one change at a time.

## 0. The shape of the problem

| | |
|---|---|
| Input | One exam (study) = several imaging series (sagittal, coronal, axial, ...) |
| Output | Probability for each of 12 findings |
| Metric | Mean AUC over the 12 findings |
| Format | Code Competition (submit a notebook, under 9 hours) |

The 12 findings:

`ACL` `MCL` `Medial Meniscus` `Lateral Meniscus` `Medial OA` `Lateral OA`
`PF OA` `Effusion` `Synovitis` `Baker's` `Contusion` `Fracture`

```
[ one exam (study) ]

  images -+- Sagittal series -+
          +- Coronal  series -+- each series = tens of slices
          +- Axial    series -+   (cross-sections through the knee)

  text ---- one radiology report written by a radiologist
            (not necessarily English: 9 languages are mixed in -> section 2)

                          |
                          v  predict

  +--------------------------------------------+
  |  probability of each of the 12 findings     |
  |  ACL / MCL / Medial Meniscus / ...          |
  +--------------------------------------------+
```

The same knee is imaged from three directions. Which finding shows up in which
direction is not arbitrary, and I measure it later (section 6).

The names are medical but they all describe "which part of the knee is damaged,
and how". One example: the **`ACL` (anterior cruciate ligament) sits in the
middle of the joint and connects the femur to the tibia**; it tears during sudden
changes of direction in sport. The label is whether that ligament is torn. The
other eleven are the same idea - a **structure** (ligament, meniscus, cartilage,
bone, joint fluid) crossed with a **kind of damage**. You don't need more anatomy
than that to follow the rest.

### What the metric decides for you

AUC only looks at **ranking**. "0.9 vs 0.8" and "0.6 vs 0.5" score the same as
long as the order is the same. Two consequences:

1. **No calibration needed.** Any monotone transform of your predictions leaves
   the score unchanged.
2. **You can blend on ranks.** Models whose outputs live on different scales can
   still be averaged fairly once converted to ranks.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")]
            if p.exists())

train = pd.read_csv(ROOT / "train.csv")
series = pd.read_csv(ROOT / "train_series.csv")

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

print(f"train   : {len(train):,} exams")
print(f"series  : {len(series):,} series")


## 1. The thing to know first: only 58 exams have labels

Open `train.csv` and it's immediately visible.

**train has 4,407 exams, but only 58 of them carry the 12 findings as labels.**
The other 4,349 come with the **radiologist's free-text report** and nothing else.

```
train 4,407 exams
  |
  +-    58 : images + report + [12 findings labelled]
  |            -> ground truth, so useful for checking labels you build
  |
  +- 4,349 : images + report + (no labels)
               |
               -> unusable for training as-is.
                  Read the report and they become usable.
```

To train on those 4,349 exams you have to **read the report text and decide the
presence/absence of each of the 12 findings yourself**. Skip that and your
training set stays at 58 exams, which is not enough to train an image model.

So before touching "predict findings from images", this competition makes you
solve **"build training labels from reports"** first. Three steps:

1. Read the reports and **build your own labels** for all 4,407 exams
2. Train an image model on those labels
3. Check label quality against the 58 ground-truth exams

The 58 exams are few, but they are the only honest check you have on step 1.


In [ ]:
# ---- how many exams actually carry labels ---------------------------------
labeled = train[TARGETS].notna().all(axis=1)
print(f"exams with all 12 findings labelled: {labeled.sum()} / {len(train):,}")
print(f"exams with a report                : {train['Report'].notna().sum():,}")

print("\n--- example report (first 200 chars) ---")
print(train.loc[train["Report"].notna(), "Report"].iloc[0][:200])


## 2. The reports are in nine languages

Not just English. English, Spanish, French, Dutch, German, Turkish, Croatian,
Greek and Russian are all present.

The design decision here is **not to detect the language and branch on it**.

The reason is simple: **if the language detection is wrong, that report becomes
unreadable entirely**. Cheap heuristics like "contains 'the', so English" fire
happily on Dutch and German text too.

Instead I build **one dictionary holding the cues for all nine languages** and
apply all of it to every report. There is no language to get wrong, so that
whole failure mode disappears.

Reading a report needs three kinds of expression:

| kind | examples |
|---|---|
| the finding itself | `anterior cruciate`, `cruzado anterior`, `vorderes kreuzband` |
| negation | `no`, `sin`, `ohne`, `unremarkable` |
| hedging | `possible`, `suspected`, `cannot exclude` |

Miss the negation and hedging and "no ACL tear" becomes "ACL tear present".


In [ ]:
# ---- which languages show up ----------------------------------------------
# No language detection (section 2). This is only to confirm it isn't all English.
CUE = {
    "English": r"\bmeniscus\b|\bligament\b",
    "Spanish": r"\bmenisco\b|\bligamento\b",
    "French":  r"\bm[eé]nisque\b|ligament crois",
    "Dutch":   r"\bkruisband\b",
    "German":  r"\bmeniskus\b|\bkreuzband\b",
    "Turkish": r"men[iı]sk[uü]s|çapraz bağ",
    "Russian": r"мениск|связк",
    "Greek":   r"μηνίσκ|σύνδεσμ",
}
rep = train["Report"].fillna("").str.lower()
for name, pat in CUE.items():
    n = rep.str.contains(pat, regex=True).sum()
    print(f"  reports containing {name:8s} cues: {n:5,}")
print("\n-> Not English-only. Apply all nine dictionaries at once, no detection.")


## 3. The image data, in numbers

| | |
|---|---|
| DICOM files | **819,078** |
| Total size | **~490 GB** |
| Median file size | 0.526 MB |

0.526 MB matches exactly what a 512x512 16-bit image takes **uncompressed**. And
indeed, all 22,859 files I sampled were uncompressed (Explicit VR Little Endian).

So decoding is cheap - there is no compression to undo. The expensive parts are
**opening a very large number of files** and **resizing**.

Which means: re-reading 490 GB every epoch is out of the question. Run the
preprocessing **once** and cache small arrays; training then only ever touches
the cache.

## 4. Check what the provided columns actually contain

`train_series.csv` describes each series with `Anatomical_Plane`,
`Fluid_Sensitive` and `Fat_Suppression`.

By their names, `Fluid_Sensitive` (does this sequence highlight fluid?) and
`Fat_Suppression` (does it null out fat?) look like **different properties**.
Physically they are independent. But count them:

**The two columns agree on all 24,371 series (100%).** One is a copy of the
other, and neither of them tells you the contrast (T1 / PD / T2).

Contrast matters - it's what makes a finding visible at all:

| | contrast | what it shows |
|---|---|---|
| TR < 800 | T1 | fat bright. anatomy |
| TE < 50 | PD | intermediate. meniscus |
| otherwise | T2 | fluid bright. effusion, inflammation |

You can also read it off the series description text, but **11.8% are anonymised
to `DummySeriesDesc!`**, so text alone classifies only 77%. TE/TR is present on
95.1% of series, so **using both gets you to 100%**.

> The 11.8% / 95.1% figures come from scanning the headers of all 24,371 series
> once. The cell below samples 300 series to keep the runtime down, so the
> percentages it prints will differ somewhat.

A provided column does not necessarily contain what its name suggests. Counting
two of them against each other is what surfaced both facts here: that one
duplicates the other, and that neither encodes T1/PD/T2. Used as features on the
strength of their names, neither would have come up.


In [ ]:
# ---- do the provided columns describe different properties? ---------------
ct = pd.crosstab(series["Fluid_Sensitive"], series["Fat_Suppression"])
print("Fluid_Sensitive x Fat_Suppression:")
print(ct)

agree = (series["Fluid_Sensitive"] == series["Fat_Suppression"]).mean()
print(f"\nfraction where the two agree: {agree:.1%}")
print("-> They say the same thing. Neither carries contrast (T1/PD/T2).")
print("   Recover it from DICOM TR/TE.")


In [ ]:
# ---- peek at DICOM headers (physical scale and contrast) ------------------
# Opening all 819,078 files is expensive, so take one file per series for a
# random 300 series. That is plenty to see the distribution.
import pydicom

sample = series.sample(min(300, len(series)), random_state=0)

rows = []
for r in sample.itertuples():
    d = ROOT / "train_series" / r.StudyInstanceUID / r.SeriesInstanceUID
    files = list(d.glob("*.dcm"))[:1] if d.exists() else []
    if not files:
        continue
    try:
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
    except Exception:
        continue
    ps = getattr(ds, "PixelSpacing", None)
    rows.append(dict(
        spacing=float(ps[0]) if ps is not None else np.nan,
        rows_px=float(getattr(ds, "Rows", np.nan)),
        TR=float(getattr(ds, "RepetitionTime", np.nan) or np.nan),
        TE=float(getattr(ds, "EchoTime", np.nan) or np.nan),
        desc=str(getattr(ds, "SeriesDescription", "")),
    ))

hdr = pd.DataFrame(rows)
print(f"series read: {len(hdr)}")
print("\nmillimetres per pixel:")
print(hdr["spacing"].describe()[["min", "25%", "50%", "75%", "max"]].round(3))
print("\n-> It varies by site and scanner. Resizing alone will not align sizes.")


In [ ]:
# ---- recover contrast from TR/TE ------------------------------------------
def contrast_of(tr, te):
    """Map a TR/TE pair to T1 / PD / T2. A simple rule is good enough."""
    if not np.isfinite(tr) or not np.isfinite(te):
        return "unknown"
    if tr < 800:
        return "T1"
    if te < 50:
        return "PD"
    return "T2"


hdr["contrast"] = [contrast_of(t, e) for t, e in zip(hdr["TR"], hdr["TE"])]
print("contrast from TR/TE:")
print(hdr["contrast"].value_counts().to_string())

dummy = hdr["desc"].str.contains("Dummy", case=False, na=False).mean()
print(f"\nfraction of series with an anonymised description: {dummy:.1%}")
print("-> Names alone cannot classify them. Combine with TR/TE.")


## 5. Aligning physical scale - at 160mm it was doing nothing 28.9% of the time

In MRI, **how many millimetres one pixel covers differs between sites and
scanners**. DICOM stores it in `PixelSpacing`.

Ignore it and "resize everything to 224x224" means **the same meniscus appears at
more than twice the size** in one image versus another. To the model those are
different objects.

So: fix a **physical field of view in millimetres** around the joint, crop that,
and resize the crop.

```
crop width (pixels) = field of view (mm) / PixelSpacing (mm/pixel)
```

I started with a 160mm field of view. It **was not working**.

Some acquisitions are narrower than 160mm to begin with. When that happens the
crop width saturates at the image size, and **the physical-scale alignment
silently does nothing** - you are just using the whole image.

Counted: at 160mm this happened on **28.9% of series**. At 130mm it drops to
0.5%. (Both from scanning every series; the cell below samples 300, so its
numbers differ slightly.)

Fixing only this one thing moved the LB from 0.857 to 0.861.

Worth stating plainly: "implemented" and "having an effect" are different
things. The way to tell them apart is to **count the cases where it didn't
apply**.


In [ ]:
# ---- what field of view should we use? ------------------------------------
# crop width = fov / spacing. Above the image size it saturates and the
# physical-scale alignment stops doing anything. Count how often, per fov.
ok = hdr.dropna(subset=["spacing", "rows_px"])
print(f"{'fov':>6s}  {'series where it no-ops':>24s}")
for fov in (160, 150, 140, 130, 120):
    need = fov / ok["spacing"]
    clipped = (need > ok["rows_px"]).mean()
    print(f"{fov:4d}mm  {clipped:23.1%}")
print("\n-> 160mm no-ops on a non-trivial fraction. I went with 130mm.")


## 6. Which plane shows which finding

A knee MRI is acquired from several directions: sagittal (S), coronal (C),
axial (A).

Rather than assume "feed it everything", I built versions with planes removed and
**submitted them**, to measure which findings depend on which plane.

| planes used | LB |
|---|---|
| coronal only (C) | 0.718 |
| sagittal only (S) | 0.772 |
| sagittal + axial (S+A) | 0.802 |
| **all three** | **0.820** |

Broken down per finding, the ordering lands where anatomy says it should. These
are **local validation AUCs per finding**, a different quantity from the LB
numbers above.

| finding | best single plane (validation AUC) | anatomical expectation |
|---|---|---|
| MCL | **coronal 0.705** (sagittal is worst at 0.477) | the MCL is read on coronal |
| Medial Meniscus | **sagittal 0.729** | menisci are read on sagittal |
| PF OA (patellofemoral) | **axial 0.629** (sagittal 0.590 / coronal 0.528) | cartilage behind the patella is axial |
| Synovitis | **axial 0.804** | joint fluid and synovium are axial |

That is evidence the model is looking somewhere anatomically sensible, rather
than latching onto a shortcut such as a site-specific imaging quirk.

If you only add one plane to sagittal, add axial.

## 7. How to split validation

Images come from many sites and scanners. If the same scanner appears on both
sides of a split, the model can learn "this is scanner X, and scanner X exams
tend to look like this", and validation flatters it.

So I build a **scanner fingerprint** from the DICOM headers
(`Manufacturer | ModelName | SoftwareVersions | MagneticFieldStrength`), which
gives 112 groups, and split with `GroupKFold` so **no group straddles the
split**. Validation is then always "a scanner the model has not seen".


In [ ]:
# ---- how to split validation ----------------------------------------------
# Group by scanner fingerprint and split so no group straddles the boundary.
# The real fingerprint comes from one pass over every series header; here we
# just demonstrate the splitting discipline.
from sklearn.model_selection import GroupKFold

demo = train[["StudyInstanceUID"]].copy()
demo["group"] = np.random.default_rng(0).integers(0, 112, len(demo))  # 112 real groups

gkf = GroupKFold(n_splits=5)
for f, (tr_i, va_i) in enumerate(gkf.split(demo, groups=demo["group"])):
    a = set(demo.iloc[tr_i]["group"])
    b = set(demo.iloc[va_i]["group"])
    assert not (a & b), "a group straddles train and validation"
    print(f"  fold {f}: train {len(tr_i):,} / valid {len(va_i):,}  overlap {len(a & b)}")
print("\n-> Validation is always an unseen scanner.")


## 8. All of that fixes the baseline

The EDA findings turn directly into settings.

| decision | because |
|---|---|
| Training target = labels built from the reports | only 58 exams are labelled (1) |
| Crop a fixed 130mm physical field of view | 160mm no-ops on 28.9% (5) |
| Use all three planes | different planes carry different findings (6) |
| GroupKFold on scanner fingerprint, 5 folds | stop the model learning the scanner (7) |
| Average folds; blend on ranks | AUC only sees ranking (0) |

What's left is the image model, and there I use a conventional setup.

### Building the input (2.5D)

MRI is 3D, but 3D CNNs are heavy and there isn't much training data. So: **2.5D**.

1. Pick one representative series per plane
2. Sample **24 slices** at even spacing
3. Take **3 consecutive slices as one group**, treated as the RGB channels (8 groups)
4. 3 planes x 8 groups = 24 "images" make up one exam

For a single plane:

```
original series (tens of slices)
  ...............................

        | pick 24 at even spacing
        v
  .  .  .  .  .  .  .  .   ... (24)

        | consecutive 3 -> RGB
        v
  [1,2,3] [4,5,6] [7,8,9] ...    8 groups

        | encoder, per group
        v
  one feature vector per group
        |
        | learn which groups matter, per finding
        v
  probability for each of the 12 findings
```

Putting neighbouring slices in the channels lets a plain 2D model see
**continuity perpendicular to the plane**. A structure like the meniscus, which
only appears on certain slices, ends up concentrated in the groups containing
those slices.

### Model

- **Encoder**: DINOv2 ViT-S/14 (self-supervised pretraining)
- **Pooling**: learn which of the 8 groups to weight **separately for each
  finding** (the best slices for the meniscus are not the best slices for the ACL)
- **Learning rate**: 1e-3 on the head, 8e-6 on the encoder, and only the **last
  6 blocks** unfrozen

That last point matters. Fine-tuning the whole pretrained encoder at a normal
learning rate **overwrites the pretrained weights and destroys the
representation**. The version that trained all layers at 1e-4 lost to a far
smaller resnet18. Adapt the encoder; don't retrain it.

## 9. What each change was worth

None of this was decided up front. I submitted one change at a time and measured
each contribution.

| change | public LB |
|---|---|
| resnet18 2.5D + weak labels from my own rules | 0.820 |
| **target replaced with LLM-read labels from the reports** | **0.857** (+0.037) |
| crop 160mm -> 130mm | 0.861 (+0.004) |
| encoder -> DINOv2 ViT-S/14 with discriminative LRs | 0.882 (+0.021) |
| pooling made per-finding | 0.884 (+0.002) |
| input resolution 224px -> **336px** | **0.898** (+0.014) |

**The biggest single move was the label swap (+0.037)** - larger than every image
modelling improvement put together. The claim in section 1, that this is a
label-building problem, shows up directly in the results.

Resolution was second (+0.014). 130mm at 336px works out to **0.387mm per
pixel**, and I read that as roughly what it takes to see a meniscal tear.

Because the changes were separated, each contribution is a number. Bundled
together, there would be no way to tell whether the gain came from the labels or
the resolution - and no basis for deciding where to spend effort next.

## 10. Where to go from here

- **Improve label quality.** It was the largest axis by a wide margin, and
  better report reading still has room.
- **Train several models and blend.** Though: blending in a model that is weak on
  its own does not help. What helps is a model that is **about as strong as your
  baseline and wrong in different places**.
- **Blend with public models.** Someone else's model trained with a different
  design differs from yours far more than diversity you manufacture in-house.


## Appendix: running this baseline end to end

Everything above is analysis. The cells below actually run the baseline described
in sections 8-9 and write `submission.csv`, so the notebook produces the **public
LB 0.898** entry rather than just describing it.

The weights are the 5 folds of that configuration, published as a CC0 dataset:
`kitopl/rsna-knee-eda-baseline-weights`. They were trained on the public
report labels from `stevenleehans/rsna-knee-llm-report-labels`, so nothing here
depends on private data.

Two things worth repeating because they are easy to get wrong:

- **Preprocessing at inference must match training exactly.** The 130mm crop, the
  24-slice sampling, the 1st/99th percentile normalisation - if any of it drifts,
  the score drops and nothing tells you why. The code below is a verbatim copy of
  the training-time preprocessing.
- **The encoder is built with `pretrained=False`.** The backbone weights are
  already inside the checkpoint, so no internet access is needed.


In [ ]:
# ---- inference config ------------------------------------------------------
import os
import time
from concurrent.futures import ProcessPoolExecutor

import cv2
import pydicom
import timm
import torch
import torch.nn as nn
from scipy.stats import rankdata

OUT = "/kaggle/working"
INPUT = str(ROOT)

PLANES_ORDER = ["Sagittal", "Coronal", "Axial"]
PLANES, N_SLICES, HW = 3, 24, 336
TRIPLETS = N_SLICES // 3
FOV_MM = 130.0          # must match training (section 5)
BACKBONE = "vit_small_patch14_dinov2.lvd142m"


def find_weights(fname="model_fold0.pt"):
    """Locate the weights by searching /kaggle/input.

    Hardcoding the mount path is brittle - it depends on how the dataset was
    attached. Search for the file instead, and print what is actually there so
    a miss is diagnosable rather than mysterious.
    """
    for root, _, files in os.walk("/kaggle/input"):
        if fname in files:
            return root
    return None

BATCH = 8
TIME_BUDGET = 7.5 * 3600   # margin against the 9h limit; fill the rest with 0.5

dev = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={dev}  torch={torch.__version__}  timm={timm.__version__}")
print("mounted under /kaggle/input:", sorted(os.listdir("/kaggle/input")))
WEIGHTS = find_weights()
print("weights dir:", WEIGHTS)
assert WEIGHTS, "model_fold0.pt not found anywhere under /kaggle/input"
print("weights files:", sorted(f for f in os.listdir(WEIGHTS) if f.endswith(".pt")))


In [ ]:
# ---- preprocessing: verbatim from training ---------------------------------
# Any divergence between this and the training-time preprocessing costs score
# silently, so it is copied rather than re-derived.

def pick_series(series_df):
    """One series per (study, plane), preferring fluid-sensitive."""
    df = series_df.copy()
    df["_pref"] = -df["Fluid_Sensitive"].fillna(0).astype(int)
    return (df.sort_values(["StudyInstanceUID", "Anatomical_Plane",
                            "_pref", "SeriesInstanceUID"])
              .groupby(["StudyInstanceUID", "Anatomical_Plane"], as_index=False)
              .head(1))


def slice_position(ds):
    """Projection of the image position onto the slice normal.

    Do not use the z component directly: for sagittal the slice axis is x and
    for coronal it is y.
    """
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    if iop is None or ipp is None:
        return None
    n = np.cross([float(v) for v in iop[:3]], [float(v) for v in iop[3:]])
    return float(np.dot([float(v) for v in ipp], n))


def load_volume(split, study, series):
    d = os.path.join(INPUT, f"{split}_series", study, series)
    try:
        paths = [f.path for f in os.scandir(d) if f.name.endswith(".dcm")]
    except OSError as e:
        return None, f"scandir: {e}"
    if not paths:
        return None, "no dcm"

    items = []
    for p in paths:
        try:
            ds = pydicom.dcmread(p, force=True)
            arr = ds.pixel_array
        except Exception:
            continue
        ps = getattr(ds, "PixelSpacing", None)
        items.append((slice_position(ds), arr,
                      float(ps[0]) if ps is not None else None))
    if not items:
        return None, "all slices failed to decode"

    if all(it[0] is None for it in items):
        items = [(i, a, s) for i, (_, a, s) in enumerate(items)]
    else:
        items = [it for it in items if it[0] is not None]
        items.sort(key=lambda x: x[0])

    out = np.zeros((N_SLICES, HW, HW), dtype=np.uint8)
    idx = np.linspace(0, len(items) - 1, N_SLICES).round().astype(int)
    for k, i in enumerate(idx):
        _, a, spacing = items[i]
        a = a.astype(np.float32)
        if a.ndim != 2:
            a = a[..., 0] if a.ndim == 3 else a.reshape(a.shape[-2:])
        if spacing and spacing > 0:
            # crop a fixed physical field of view, then resize (section 5)
            crop_px = int(round(FOV_MM / spacing))
            crop_px = max(16, min(crop_px, min(a.shape)))
            cy, cx = a.shape[0] // 2, a.shape[1] // 2
            h = crop_px // 2
            a = a[max(0, cy - h):cy + h, max(0, cx - h):cx + h]
        flat = a[::4, ::4].ravel()
        lo, hi = np.percentile(flat, [1, 99]) if flat.size else (0.0, 1.0)
        a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
        out[k] = (cv2.resize(a, (HW, HW),
                             interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
    return out, None


def build_study(args):
    study, chosen = args
    vol = np.zeros((PLANES, N_SLICES, HW, HW), dtype=np.uint8)
    for pi, plane in enumerate(PLANES_ORDER):
        sid = chosen.get(plane)
        if sid is None:
            continue
        v, _ = load_volume("test", study, sid)
        if v is not None:
            vol[pi] = v
    return study, vol


In [ ]:
# ---- model: must match training token for token -----------------------------
# If this drifts from the training definition, the state_dict keys or shapes
# stop lining up and inference breaks quietly.

PRETRAINED = False       # backbone weights live in the checkpoint
USE_TRIPLET_POS = False


def build_encoder(backbone):
    kw = dict(pretrained=PRETRAINED, num_classes=0, in_chans=3)
    if backbone.startswith("vit_"):
        # DINOv2 is patch14 and pretrained at 518px; timm resamples the
        # positional embedding when img_size is given.
        assert HW % 14 == 0, f"patch14 ViT needs HW divisible by 14: {HW}"
        kw["img_size"] = HW
    return timm.create_model(backbone, **kw)


class Net(nn.Module):
    """2.5D encoder with per-finding attention over triplets.

    The attention produces one weight per (plane, triplet, finding), so the
    slices that matter for the meniscus can differ from the ones that matter
    for the ACL. Collapsing to a single vector before the classifier, as an
    earlier version did, throws that away.
    """

    def __init__(self, backbone, n_out=len(TARGETS)):
        super().__init__()
        self.enc = build_encoder(backbone)
        d = self.enc.num_features
        self.n_out = n_out
        self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, n_out))
        self.pos = nn.Parameter(torch.zeros(1, 1, TRIPLETS, d))
        self.drop = nn.Dropout(0.2)
        self.cls_w = nn.Parameter(torch.empty(n_out, PLANES * d))
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        nn.init.normal_(self.cls_w, std=0.02)

    def forward(self, x):                      # (B, 3, 24, H, W)
        B = x.shape[0]
        x = x.reshape(B * PLANES * TRIPLETS, 3, HW, HW)
        f = self.enc(x).reshape(B, PLANES, TRIPLETS, -1)
        if USE_TRIPLET_POS:
            f = f + self.pos
        w = torch.softmax(self.att(f), dim=2)          # normalise over triplets
        p = torch.einsum("bptd,bptc->bpcd", f, w)      # per-finding plane embedding
        p = self.drop(p.permute(0, 2, 1, 3).reshape(B, self.n_out, -1))
        return (p * self.cls_w.unsqueeze(0)).sum(-1) + self.cls_b


nets = []
for f in range(5):
    p = os.path.join(WEIGHTS, f"model_fold{f}.pt")
    if not os.path.exists(p):
        continue
    m = Net(BACKBONE).to(dev).eval()
    m.load_state_dict(torch.load(p, map_location=dev))
    nets.append(m)
assert nets, f"no weights loaded from {WEIGHTS}"
print(f"loaded {len(nets)} folds of {BACKBONE}")


In [ ]:
# ---- run inference and write submission.csv --------------------------------
t0 = time.time()

test = pd.read_csv(f"{INPUT}/test.csv")
te_series = pd.read_csv(f"{INPUT}/test_series.csv")
print(f"test: {len(test):,} studies / {len(te_series):,} series")

sel = pick_series(te_series)
tasks = [(s, dict(zip(g.Anatomical_Plane, g.SeriesInstanceUID)))
         for s, g in sel.groupby("StudyInstanceUID")]
print(f"selected {len(sel):,} series for {len(tasks):,} studies")

order, chunks = [], []
workers = max(1, (os.cpu_count() or 4))
with ProcessPoolExecutor(max_workers=workers) as ex:
    buf_u, buf_x = [], []
    for i, (study, vol) in enumerate(ex.map(build_study, tasks, chunksize=4), 1):
        buf_u.append(study)
        buf_x.append(vol)
        last = (i == len(tasks))
        if len(buf_u) < BATCH and not last:
            continue

        x = torch.from_numpy(np.stack(buf_x)).float().div_(255.0)
        x = ((x - 0.449) / 0.226).to(dev)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(dev == "cuda")):
            # average the 5 folds in probability space, as at training time
            p = sum(torch.sigmoid(m(x)) for m in nets) / len(nets)
        chunks.append(p.float().cpu().numpy())
        order += buf_u
        buf_u, buf_x = [], []

        if i % 200 == 0 or last:
            el = time.time() - t0
            print(f"    {i:,}/{len(tasks):,}  {el/60:.1f} min  "
                  f"eta {(len(tasks)-i)*el/max(i,1)/60:.0f} min", flush=True)
        if time.time() - t0 > TIME_BUDGET:
            print("[budget] out of time; remaining studies get 0.5")
            break

P = np.concatenate(chunks) if chunks else np.zeros((0, len(TARGETS)))
n = len(order)
assert len(P) == n, f"{len(P)} != {n}"

# AUC only sees ranking, so convert to per-column ranks before writing.
# With a single model this is a monotone transform and does not change the
# score; it keeps the code identical to the multi-model blend.
pred = np.column_stack([rankdata(P[:, j]) / max(n, 1) for j in range(len(TARGETS))])
preds = dict(zip(order, pred))

sub = pd.DataFrame({"StudyInstanceUID": test.StudyInstanceUID})
arr = np.full((len(sub), len(TARGETS)), 0.5, np.float64)
for i, u in enumerate(sub.StudyInstanceUID):
    if u in preds:
        arr[i] = preds[u]
for j, t in enumerate(TARGETS):
    sub[t] = arr[:, j]

sample = pd.read_csv(f"{INPUT}/sample_submission.csv")
sub = sub[list(sample.columns)]
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(test)
assert sub[TARGETS].notna().all().all()
sub.to_csv(f"{OUT}/submission.csv", index=False)

n_pred = sum(1 for u in sub.StudyInstanceUID if u in preds)
print(f"\nwrote submission.csv {sub.shape}  predicted {n_pred:,}/{len(sub):,} "
      f"(filled with 0.5: {len(sub)-n_pred:,})")
print(f"done in {(time.time()-t0)/60:.1f} min")
